In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/chillalyash/h1n1-vaccine-dataset/training_set_features.csv
/kaggle/input/datasets/chillalyash/h1n1-vaccine-dataset/test_set_features.csv
/kaggle/input/datasets/chillalyash/h1n1-vaccine-dataset/training_set_labels.csv
/kaggle/input/datasets/chillalyash/h1n1-vaccine-dataset/submission_format.csv


In [1]:
import pandas as pd
submission_file = pd.read_csv("/kaggle/input/datasets/chillalyash/h1n1-vaccine-dataset/submission_format.csv")
submission_file.head(10)

,respondent_id,h1n1_vaccine,seasonal_vaccine
0,26707,0.5,0.7
1,26708,0.5,0.7
2,26709,0.5,0.7
3,26710,0.5,0.7
4,26711,0.5,0.7
5,26712,0.5,0.7
6,26713,0.5,0.7
7,26714,0.5,0.7
8,26715,0.5,0.7
9,26716,0.5,0.7


In [2]:
train_feature = pd.read_csv("/kaggle/input/datasets/chillalyash/h1n1-vaccine-dataset/training_set_features.csv")
train_label = pd.read_csv("/kaggle/input/datasets/chillalyash/h1n1-vaccine-dataset/training_set_labels.csv")
test_df = pd.read_csv("/kaggle/input/datasets/chillalyash/h1n1-vaccine-dataset/test_set_features.csv")

In [4]:
train_feature.head()

,respondent_id,h1n1_concern,h1n1_knowledge,behavioral_antiviral_meds,behavioral_avoidance,behavioral_face_mask,behavioral_wash_hands,behavioral_large_gatherings,behavioral_outside_home,behavioral_touch_face,...,income_poverty,marital_status,rent_or_own,employment_status,hhs_geo_region,census_msa,household_adults,household_children,employment_industry,employment_occupation
0,0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,...,Below Poverty,Not Married,Own,Not in Labor Force,oxchjgsf,Non-MSA,0.0,0.0,NaN,NaN
1,1,3.0,2.0,0.0,1.0,0.0,1.0,0.0,1.0,1.0,...,Below Poverty,Not Married,Rent,Employed,bhuqouqj,"MSA, Not Principle City",0.0,0.0,pxcmvdjn,xgwztkwe
2,2,1.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,"<= $75,000, Above Poverty",Not Married,Own,Employed,qufhixun,"MSA, Not Principle City",2.0,0.0,rucpziij,xtkaffoo
3,3,1.0,1.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,...,Below Poverty,Not Married,Rent,Not in Labor Force,lrircsnp,"MSA, Principle City",0.0,0.0,NaN,NaN
4,4,2.0,1.0,0.0,1.0,0.0,1.0,1.0,0.0,1.0,...,"<= $75,000, Above Poverty",Married,Own,Employed,qufhixun,"MSA, Not Principle City",1.0,0.0,wxleyezf,emcorrxb


In [5]:
train_label.head()

,respondent_id,h1n1_vaccine,seasonal_vaccine
0,0,0,0
1,1,0,1
2,2,0,0
3,3,0,1
4,4,0,0


In [6]:
train_feature.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 26707 entries, 0 to 26706
Data columns (total 36 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   respondent_id                26707 non-null  int64  
 1   h1n1_concern                 26615 non-null  float64
 2   h1n1_knowledge               26591 non-null  float64
 3   behavioral_antiviral_meds    26636 non-null  float64
 4   behavioral_avoidance         26499 non-null  float64
 5   behavioral_face_mask         26688 non-null  float64
 6   behavioral_wash_hands        26665 non-null  float64
 7   behavioral_large_gatherings  26620 non-null  float64
 8   behavioral_outside_home      26625 non-null  float64
 9   behavioral_touch_face        26579 non-null  float64
 10  doctor_recc_h1n1             24547 non-null  float64
 11  doctor_recc_seasonal         24547 non-null  float64
 12  chronic_med_condition        25736 non-null  float64
 13  child_under_6_mo

In [7]:
for cols in train_feature.columns:
    if train_feature[cols].dtypes == "object":
        print(f"{cols}: {train_feature[cols].unique()}\n")

age_group: ['55 - 64 Years' '35 - 44 Years' '18 - 34 Years' '65+ Years'
 '45 - 54 Years']

education: ['< 12 Years' '12 Years' 'College Graduate' 'Some College' nan]

race: ['White' 'Black' 'Other or Multiple' 'Hispanic']

sex: ['Female' 'Male']

income_poverty: ['Below Poverty' '<= $75,000, Above Poverty' '> $75,000' nan]

marital_status: ['Not Married' 'Married' nan]

rent_or_own: ['Own' 'Rent' nan]

employment_status: ['Not in Labor Force' 'Employed' 'Unemployed' nan]

hhs_geo_region: ['oxchjgsf' 'bhuqouqj' 'qufhixun' 'lrircsnp' 'atmpeygn' 'lzgpxyit'
 'fpwskwrf' 'mlyzmhmf' 'dqpwygqj' 'kbazzjca']

census_msa: ['Non-MSA' 'MSA, Not Principle  City' 'MSA, Principle City']

employment_industry: [nan 'pxcmvdjn' 'rucpziij' 'wxleyezf' 'saaquncn' 'xicduogh' 'ldnlellj'
 'wlfvacwt' 'nduyfdeo' 'fcxhlnwr' 'vjjrobsf' 'arjwrbjb' 'atmlpfrs'
 'msuufmds' 'xqicxuve' 'phxvnwax' 'dotnnunm' 'mfikgejo' 'cfqqtusy'
 'mcubkhph' 'haxffmxo' 'qnlwzans']

employment_occupation: [nan 'xgwztkwe' 'xtkaffoo' 'emcorr

In [8]:
df = train_feature.copy()

In [9]:
df['age_group'] = df['age_group'].map({
    '55 - 64 Years':'adults',
    '65+ Years':'senior',
    '18 - 34 Years':'young adults',
    '35 - 44 Years':'adults',
    '45 - 54 Years':'adults'
})

df['age_group'].value_counts()

age_group
adults          14649
senior           6843
young adults     5215
Name: count, dtype: int64

In [10]:
df['education'] = df['education'].replace("< 12 Years", "below 12 years")
df['education'].unique()

array(['below 12 years', '12 Years', 'College Graduate', 'Some College',
       nan], dtype=object)

In [11]:
df['education'].isnull().sum()

np.int64(1407)

In [12]:
df['education'] = df['education'].fillna('unknowm')
df['education'].value_counts()

education
College Graduate    10097
Some College         7043
12 Years             5797
below 12 years       2363
unknowm              1407
Name: count, dtype: int64

In [13]:
df['income_poverty'].value_counts()

income_poverty
<= $75,000, Above Poverty    12777
> $75,000                     6810
Below Poverty                 2697
Name: count, dtype: int64

In [14]:
# map the income poverty
df['income_order'] = df['income_poverty'].map({
    "Below Poverty":0,
    "<= $75,000, Above Poverty":1,
    "> $75,000":2
})
# Fill NaN values in 'income_order' with -1 before converting to int64
df['income_order'] = df['income_order'].fillna(df['income_order'].mode())

In [15]:
df['income_order'].value_counts()

income_order
1.0    12777
2.0     6810
0.0     2697
Name: count, dtype: int64

In [16]:
df['income_order'].isnull().sum()

np.int64(4423)

In [17]:
df['income_order'] = df['income_order'].fillna(-1)
df['income_order'].value_counts()

income_order
 1.0    12777
 2.0     6810
-1.0     4423
 0.0     2697
Name: count, dtype: int64

In [18]:
df['income_order'].isnull().sum()

np.int64(0)

In [19]:
df = df.drop('income_poverty', axis=1)
df.columns

Index(['respondent_id', 'h1n1_concern', 'h1n1_knowledge',
       'behavioral_antiviral_meds', 'behavioral_avoidance',
       'behavioral_face_mask', 'behavioral_wash_hands',
       'behavioral_large_gatherings', 'behavioral_outside_home',
       'behavioral_touch_face', 'doctor_recc_h1n1', 'doctor_recc_seasonal',
       'chronic_med_condition', 'child_under_6_months', 'health_worker',
       'health_insurance', 'opinion_h1n1_vacc_effective', 'opinion_h1n1_risk',
       'opinion_h1n1_sick_from_vacc', 'opinion_seas_vacc_effective',
       'opinion_seas_risk', 'opinion_seas_sick_from_vacc', 'age_group',
       'education', 'race', 'sex', 'marital_status', 'rent_or_own',
       'employment_status', 'hhs_geo_region', 'census_msa', 'household_adults',
       'household_children', 'employment_industry', 'employment_occupation',
       'income_order'],
      dtype='object')

In [20]:
df = df.drop(['employment_occupation','employment_industry', 'hhs_geo_region'], axis=1)
df.columns

Index(['respondent_id', 'h1n1_concern', 'h1n1_knowledge',
       'behavioral_antiviral_meds', 'behavioral_avoidance',
       'behavioral_face_mask', 'behavioral_wash_hands',
       'behavioral_large_gatherings', 'behavioral_outside_home',
       'behavioral_touch_face', 'doctor_recc_h1n1', 'doctor_recc_seasonal',
       'chronic_med_condition', 'child_under_6_months', 'health_worker',
       'health_insurance', 'opinion_h1n1_vacc_effective', 'opinion_h1n1_risk',
       'opinion_h1n1_sick_from_vacc', 'opinion_seas_vacc_effective',
       'opinion_seas_risk', 'opinion_seas_sick_from_vacc', 'age_group',
       'education', 'race', 'sex', 'marital_status', 'rent_or_own',
       'employment_status', 'census_msa', 'household_adults',
       'household_children', 'income_order'],
      dtype='object')

In [21]:
for cols in df.columns:
    if df[cols].dtypes == "object":
        print(f"{cols}: {df[cols].unique()} \n")

age_group: ['adults' 'young adults' 'senior'] 

education: ['below 12 years' '12 Years' 'College Graduate' 'Some College' 'unknowm'] 

race: ['White' 'Black' 'Other or Multiple' 'Hispanic'] 

sex: ['Female' 'Male'] 

marital_status: ['Not Married' 'Married' nan] 

rent_or_own: ['Own' 'Rent' nan] 

employment_status: ['Not in Labor Force' 'Employed' 'Unemployed' nan] 

census_msa: ['Non-MSA' 'MSA, Not Principle  City' 'MSA, Principle City'] 



In [22]:
df['race'].value_counts()

race
White                21222
Black                 2118
Hispanic              1755
Other or Multiple     1612
Name: count, dtype: int64

In [23]:
df['race'].isnull().sum()

np.int64(0)

In [24]:
df['race'] = df['race'].replace('Other or Multiple', 'Other')
df['race'].unique()

array(['White', 'Black', 'Other', 'Hispanic'], dtype=object)

In [25]:
df['rent_or_own'].isnull().sum()

np.int64(2042)

In [26]:
df['rent_or_own'] = df['rent_or_own'].fillna('rent_or_own')

In [27]:
df['rent_or_own'].isnull().sum()

np.int64(0)

In [28]:
df['rent_or_own'].value_counts()

rent_or_own
Own            18736
Rent            5929
rent_or_own     2042
Name: count, dtype: int64

In [29]:
df['rent_or_own'] = df['rent_or_own'].replace('rent_or_own','unknown')
df['rent_or_own'].value_counts()

rent_or_own
Own        18736
Rent        5929
unknown     2042
Name: count, dtype: int64

In [30]:
df['employment_status'] = df['employment_status'].replace('Not in Labor Force','Inactive')
df['employment_status'] = df['employment_status'].fillna("unknown")
df['employment_status'].value_counts()

employment_status
Employed      13560
Inactive      10231
unknown        1463
Unemployed     1453
Name: count, dtype: int64

In [31]:
df['employment_status'].isnull().sum()

np.int64(0)

In [33]:
df['marital_status'] = df['marital_status'].fillna('unkown')
df['marital_status'].value_counts()

marital_status
Married        13555
Not Married    11744
unkown          1408
Name: count, dtype: int64

In [34]:
df['marital_status'].isnull().sum()

np.int64(0)

In [35]:
for cols in df.columns:
    if df[cols].dtypes == "object":
        print(f"{cols}: {df[cols].unique()} \n")

age_group: ['adults' 'young adults' 'senior'] 

education: ['below 12 years' '12 Years' 'College Graduate' 'Some College' 'unknowm'] 

race: ['White' 'Black' 'Other' 'Hispanic'] 

sex: ['Female' 'Male'] 

marital_status: ['Not Married' 'Married' 'unkown'] 

rent_or_own: ['Own' 'Rent' 'unknown'] 

employment_status: ['Inactive' 'Employed' 'Unemployed' 'unknown'] 

census_msa: ['Non-MSA' 'MSA, Not Principle  City' 'MSA, Principle City'] 



In [36]:
for cols in df.columns:
    if df[cols].dtypes == "object":
        print(f"{cols} : {df[cols].isnull().sum()} \n")

age_group : 0 

education : 0 

race : 0 

sex : 0 

marital_status : 0 

rent_or_own : 0 

employment_status : 0 

census_msa : 0 



In [37]:
num_cols = df.select_dtypes(include=['int64', 'float64']).columns.to_list()
cat_cols = df.select_dtypes(exclude=['int64','float64']).columns.to_list()

In [39]:
print("num cols = \n", num_cols)
print("\ntotal num cols = ", len(num_cols))
print("\n cat cols = \n", cat_cols)
print("\ntotal cat cols =", len(cat_cols))

num cols = 
 ['respondent_id', 'h1n1_concern', 'h1n1_knowledge', 'behavioral_antiviral_meds', 'behavioral_avoidance', 'behavioral_face_mask', 'behavioral_wash_hands', 'behavioral_large_gatherings', 'behavioral_outside_home', 'behavioral_touch_face', 'doctor_recc_h1n1', 'doctor_recc_seasonal', 'chronic_med_condition', 'child_under_6_months', 'health_worker', 'health_insurance', 'opinion_h1n1_vacc_effective', 'opinion_h1n1_risk', 'opinion_h1n1_sick_from_vacc', 'opinion_seas_vacc_effective', 'opinion_seas_risk', 'opinion_seas_sick_from_vacc', 'household_adults', 'household_children', 'income_order']

total num cols =  25

 cat cols = 
 ['age_group', 'education', 'race', 'sex', 'marital_status', 'rent_or_own', 'employment_status', 'census_msa']

total cat cols = 8


In [40]:
df = df.drop('respondent_id', axis=1)
num_cols.remove('respondent_id')

In [45]:
df[num_cols].isnull().sum()
total_null = df[num_cols].isnull().sum().sum()
row, col = df[num_cols].shape
total = row*col
print(f"percentage null =  {(total_null / total) * 100:.2f} %")

percentage null =  3.62 %


In [46]:
df[num_cols].isnull().sum()

h1n1_concern                      92
h1n1_knowledge                   116
behavioral_antiviral_meds         71
behavioral_avoidance             208
behavioral_face_mask              19
behavioral_wash_hands             42
behavioral_large_gatherings       87
behavioral_outside_home           82
behavioral_touch_face            128
doctor_recc_h1n1                2160
doctor_recc_seasonal            2160
chronic_med_condition            971
child_under_6_months             820
health_worker                    804
health_insurance               12274
opinion_h1n1_vacc_effective      391
opinion_h1n1_risk                388
opinion_h1n1_sick_from_vacc      395
opinion_seas_vacc_effective      462
opinion_seas_risk                514
opinion_seas_sick_from_vacc      537
household_adults                 249
household_children               249
income_order                       0
dtype: int64

In [47]:
df.shape

(26707, 32)

In [48]:
# save partially cleaned data
df.to_csv("cleaned_trained_v1.csv", index=False)
print("successfully saved")

successfully saved


In [23]:
# load clened data 
cleaned_df = pd.read_csv('/kaggle/input/datasets/chillalyash/cleaned-h1n1-dataset/cleaned_trained_v1.csv')
cleaned_df.head()

,h1n1_concern,h1n1_knowledge,behavioral_antiviral_meds,behavioral_avoidance,behavioral_face_mask,behavioral_wash_hands,behavioral_large_gatherings,behavioral_outside_home,behavioral_touch_face,doctor_recc_h1n1,...,education,race,sex,marital_status,rent_or_own,employment_status,census_msa,household_adults,household_children,income_order
0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,...,below 12 years,White,Female,Not Married,Own,Inactive,Non-MSA,0.0,0.0,0.0
1,3.0,2.0,0.0,1.0,0.0,1.0,0.0,1.0,1.0,0.0,...,12 Years,White,Male,Not Married,Rent,Employed,"MSA, Not Principle City",0.0,0.0,0.0
2,1.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,NaN,...,College Graduate,White,Male,Not Married,Own,Employed,"MSA, Not Principle City",2.0,0.0,1.0
3,1.0,1.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,...,12 Years,White,Female,Not Married,Rent,Inactive,"MSA, Principle City",0.0,0.0,0.0
4,2.0,1.0,0.0,1.0,0.0,1.0,1.0,0.0,1.0,0.0,...,Some College,White,Female,Married,Own,Employed,"MSA, Not Principle City",1.0,0.0,1.0


In [3]:
cleaned_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 26707 entries, 0 to 26706
Data columns (total 32 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   h1n1_concern                 26615 non-null  float64
 1   h1n1_knowledge               26591 non-null  float64
 2   behavioral_antiviral_meds    26636 non-null  float64
 3   behavioral_avoidance         26499 non-null  float64
 4   behavioral_face_mask         26688 non-null  float64
 5   behavioral_wash_hands        26665 non-null  float64
 6   behavioral_large_gatherings  26620 non-null  float64
 7   behavioral_outside_home      26625 non-null  float64
 8   behavioral_touch_face        26579 non-null  float64
 9   doctor_recc_h1n1             24547 non-null  float64
 10  doctor_recc_seasonal         24547 non-null  float64
 11  chronic_med_condition        25736 non-null  float64
 12  child_under_6_months         25887 non-null  float64
 13  health_worker   

In [4]:
cat_cols = []
num_cols = []

for cols in cleaned_df.columns:
    if cleaned_df[cols].dtypes == "object":
        cat_cols.append(cols)
    else:
        num_cols.append(cols)

print("categorical cols = \n", cat_cols)
print("\ntotal cat cols=", len(cat_cols))
print("numerical cols = \n", num_cols)
print("\ntotal num cols =", len(num_cols))

categorical cols = 
 ['age_group', 'education', 'race', 'sex', 'marital_status', 'rent_or_own', 'employment_status', 'census_msa']

total cat cols= 8
numerical cols = 
 ['h1n1_concern', 'h1n1_knowledge', 'behavioral_antiviral_meds', 'behavioral_avoidance', 'behavioral_face_mask', 'behavioral_wash_hands', 'behavioral_large_gatherings', 'behavioral_outside_home', 'behavioral_touch_face', 'doctor_recc_h1n1', 'doctor_recc_seasonal', 'chronic_med_condition', 'child_under_6_months', 'health_worker', 'health_insurance', 'opinion_h1n1_vacc_effective', 'opinion_h1n1_risk', 'opinion_h1n1_sick_from_vacc', 'opinion_seas_vacc_effective', 'opinion_seas_risk', 'opinion_seas_sick_from_vacc', 'household_adults', 'household_children', 'income_order']

total num cols = 24


In [5]:
for cols in num_cols:
    print(f"{cols} : {cleaned_df[cols].unique()} \n")

h1n1_concern : [ 1.  3.  2.  0. nan] 

h1n1_knowledge : [ 0.  2.  1. nan] 

behavioral_antiviral_meds : [ 0.  1. nan] 

behavioral_avoidance : [ 0.  1. nan] 

behavioral_face_mask : [ 0.  1. nan] 

behavioral_wash_hands : [ 0.  1. nan] 

behavioral_large_gatherings : [ 0.  1. nan] 

behavioral_outside_home : [ 1.  0. nan] 

behavioral_touch_face : [ 1.  0. nan] 

doctor_recc_h1n1 : [ 0. nan  1.] 

doctor_recc_seasonal : [ 0. nan  1.] 

chronic_med_condition : [ 0.  1. nan] 

child_under_6_months : [ 0.  1. nan] 

health_worker : [ 0.  1. nan] 

health_insurance : [ 1. nan  0.] 

opinion_h1n1_vacc_effective : [ 3.  5.  4.  2.  1. nan] 

opinion_h1n1_risk : [ 1.  4.  3.  2.  5. nan] 

opinion_h1n1_sick_from_vacc : [ 2.  4.  1.  5.  3. nan] 

opinion_seas_vacc_effective : [ 2.  4.  5.  3.  1. nan] 

opinion_seas_risk : [ 1.  2.  4.  3.  5. nan] 

opinion_seas_sick_from_vacc : [ 2.  4.  1.  5. nan  3.] 

household_adults : [ 0.  2.  1.  3. nan] 

household_children : [ 0.  3.  2.  1. nan] 

In [9]:
train_label.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 26707 entries, 0 to 26706
Data columns (total 3 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   respondent_id     26707 non-null  int64
 1   h1n1_vaccine      26707 non-null  int64
 2   seasonal_vaccine  26707 non-null  int64
dtypes: int64(3)
memory usage: 626.1 KB


In [3]:
test_df.head()

,respondent_id,h1n1_concern,h1n1_knowledge,behavioral_antiviral_meds,behavioral_avoidance,behavioral_face_mask,behavioral_wash_hands,behavioral_large_gatherings,behavioral_outside_home,behavioral_touch_face,...,income_poverty,marital_status,rent_or_own,employment_status,hhs_geo_region,census_msa,household_adults,household_children,employment_industry,employment_occupation
0,26707,2.0,2.0,0.0,1.0,0.0,1.0,1.0,0.0,1.0,...,"> $75,000",Not Married,Rent,Employed,mlyzmhmf,"MSA, Not Principle City",1.0,0.0,atmlpfrs,hfxkjkmi
1,26708,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,Below Poverty,Not Married,Rent,Employed,bhuqouqj,Non-MSA,3.0,0.0,atmlpfrs,xqwwgdyp
2,26709,2.0,2.0,0.0,0.0,1.0,1.0,1.0,1.0,1.0,...,"> $75,000",Married,Own,Employed,lrircsnp,Non-MSA,1.0,0.0,nduyfdeo,pvmttkik
3,26710,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,"<= $75,000, Above Poverty",Married,Own,Not in Labor Force,lrircsnp,"MSA, Not Principle City",1.0,0.0,NaN,NaN
4,26711,3.0,1.0,1.0,1.0,0.0,1.0,1.0,1.0,1.0,...,"<= $75,000, Above Poverty",Not Married,Own,Employed,lzgpxyit,Non-MSA,0.0,1.0,fcxhlnwr,mxkfnird


In [4]:
test_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 26708 entries, 0 to 26707
Data columns (total 36 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   respondent_id                26708 non-null  int64  
 1   h1n1_concern                 26623 non-null  float64
 2   h1n1_knowledge               26586 non-null  float64
 3   behavioral_antiviral_meds    26629 non-null  float64
 4   behavioral_avoidance         26495 non-null  float64
 5   behavioral_face_mask         26689 non-null  float64
 6   behavioral_wash_hands        26668 non-null  float64
 7   behavioral_large_gatherings  26636 non-null  float64
 8   behavioral_outside_home      26626 non-null  float64
 9   behavioral_touch_face        26580 non-null  float64
 10  doctor_recc_h1n1             24548 non-null  float64
 11  doctor_recc_seasonal         24548 non-null  float64
 12  chronic_med_condition        25776 non-null  float64
 13  child_under_6_mo

In [8]:
cat_cols = []
for cols in test_df.columns:
    if test_df[cols].dtypes == "object":
        cat_cols.append(cols)
cat_cols

['age_group',
 'education',
 'race',
 'sex',
 'income_poverty',
 'marital_status',
 'rent_or_own',
 'employment_status',
 'hhs_geo_region',
 'census_msa',
 'employment_industry',
 'employment_occupation']

In [9]:
for cols in cat_cols:
    print(f"{cols} : {test_df[cols].unique()}\n")

age_group : ['35 - 44 Years' '18 - 34 Years' '55 - 64 Years' '65+ Years'
 '45 - 54 Years']

education : ['College Graduate' '12 Years' 'Some College' '< 12 Years' nan]

race : ['Hispanic' 'White' 'Black' 'Other or Multiple']

sex : ['Female' 'Male']

income_poverty : ['> $75,000' 'Below Poverty' '<= $75,000, Above Poverty' nan]

marital_status : ['Not Married' 'Married' nan]

rent_or_own : ['Rent' 'Own' nan]

employment_status : ['Employed' 'Not in Labor Force' 'Unemployed' nan]

hhs_geo_region : ['mlyzmhmf' 'bhuqouqj' 'lrircsnp' 'lzgpxyit' 'fpwskwrf' 'oxchjgsf'
 'dqpwygqj' 'qufhixun' 'kbazzjca' 'atmpeygn']

census_msa : ['MSA, Not Principle  City' 'Non-MSA' 'MSA, Principle City']

employment_industry : ['atmlpfrs' 'nduyfdeo' nan 'fcxhlnwr' 'pxcmvdjn' 'arjwrbjb' 'mfikgejo'
 'rucpziij' 'wxleyezf' 'haxffmxo' 'ldnlellj' 'vjjrobsf' 'cfqqtusy'
 'xicduogh' 'dotnnunm' 'xqicxuve' 'wlfvacwt' 'saaquncn' 'msuufmds'
 'mcubkhph' 'phxvnwax' 'qnlwzans']

employment_occupation : ['hfxkjkmi' 'xqwwgdyp'

In [10]:
test_df = test_df.drop(['hhs_geo_region','employment_industry','employment_occupation'], axis=1)
print("unnecessary columns dropped successfully!")
cols_to_remove = ['hhs_geo_region','employment_industry','employment_occupation']
for cols in cols_to_remove:
    cat_cols.remove(cols)
cat_cols

unnecessary columns dropped successfully!


['age_group',
 'education',
 'race',
 'sex',
 'income_poverty',
 'marital_status',
 'rent_or_own',
 'employment_status',
 'census_msa']

In [11]:
test_df['age_group'] = test_df['age_group'].map({
    '55 - 64 Years':'adults',
    '65+ Years':'senior',
    '18 - 34 Years':'young adults',
    '35 - 44 Years':'adults',
    '45 - 54 Years':'adults'
})

test_df['education'] = test_df['education'].replace("< 12 Years", "below 12 years")

test_df['income_order'] = test_df['income_poverty'].map({
    "Below Poverty":0,
    "<= $75,000, Above Poverty":1,
    "> $75,000":2
})

test_df['race'] = test_df['race'].replace('Other or Multiple', 'Other')

In [12]:
for cols in ['age_group','education','income_order','race']:
    print(f"{cols} : {test_df[cols].unique()}\n")

age_group : ['adults' 'young adults' 'senior']

education : ['College Graduate' '12 Years' 'Some College' 'below 12 years' nan]

income_order : [ 2.  0.  1. nan]

race : ['Hispanic' 'White' 'Black' 'Other']



In [18]:
test_df = test_df.drop('income_poverty', axis=1)
cat_cols.remove('income_poverty')

In [19]:
test_df.isnull().sum()

respondent_id                      0
h1n1_concern                      85
h1n1_knowledge                   122
behavioral_antiviral_meds         79
behavioral_avoidance             213
behavioral_face_mask              19
behavioral_wash_hands             40
behavioral_large_gatherings       72
behavioral_outside_home           82
behavioral_touch_face            128
doctor_recc_h1n1                2160
doctor_recc_seasonal            2160
chronic_med_condition            932
child_under_6_months             813
health_worker                    789
health_insurance               12228
opinion_h1n1_vacc_effective      398
opinion_h1n1_risk                380
opinion_h1n1_sick_from_vacc      375
opinion_seas_vacc_effective      452
opinion_seas_risk                499
opinion_seas_sick_from_vacc      521
age_group                          0
education                       1407
race                               0
sex                                0
marital_status                  1442
r

In [21]:
test_df['employment_status'] = test_df['employment_status'].replace('Not in Labor Force','Inactive')
test_df['employment_status'] = test_df['employment_status'].fillna("unknown")

test_df['income_order'] = test_df['income_order'].fillna(-1)

In [22]:
for  cols in cat_cols:
    print(f"{cols} : {test_df[cols].unique()}\n")

age_group : ['adults' 'young adults' 'senior']

education : ['College Graduate' '12 Years' 'Some College' 'below 12 years' nan]

race : ['Hispanic' 'White' 'Black' 'Other']

sex : ['Female' 'Male']

marital_status : ['Not Married' 'Married' nan]

rent_or_own : ['Rent' 'Own' nan]

employment_status : ['Employed' 'Inactive' 'Unemployed' 'unknown']

census_msa : ['MSA, Not Principle  City' 'Non-MSA' 'MSA, Principle City']



In [24]:
test_df['rent_or_own'] = test_df['rent_or_own'].fillna('unknown')
test_df['marital_status'] = test_df['marital_status'].fillna('unknown')
test_df['education'] = test_df['education'].fillna('unknown')

In [29]:
for cols in cat_cols:
    print(f"{cols} : {test_df[cols].unique()}\n")

age_group : ['adults' 'young adults' 'senior']

education : ['College Graduate' '12 Years' 'Some College' 'below 12 years' 'unknown']

race : ['Hispanic' 'White' 'Black' 'Other']

sex : ['Female' 'Male']

marital_status : ['Not Married' 'Married' 'unknown']

rent_or_own : ['Rent' 'Own' 'unknown']

employment_status : ['Employed' 'Inactive' 'Unemployed' 'unknown']

census_msa : ['MSA, Not Principle  City' 'Non-MSA' 'MSA, Principle City']



In [30]:
for cols in cat_cols:
    print(f"{cols} : {cleaned_df[cols].unique()}\n")

age_group : ['adults' 'young adults' 'senior']

education : ['below 12 years' '12 Years' 'College Graduate' 'Some College' 'unknowm']

race : ['White' 'Black' 'Other' 'Hispanic']

sex : ['Female' 'Male']

marital_status : ['Not Married' 'Married' 'unkown']

rent_or_own : ['Own' 'Rent' 'unknown']

employment_status : ['Inactive' 'Employed' 'Unemployed' 'unknown']

census_msa : ['Non-MSA' 'MSA, Not Principle  City' 'MSA, Principle City']



In [18]:
cat_preprocessor = Pipeline(steps=[
    ('cat',cat_cols),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

num_preprocessor = Pipeline(steps=[
    ('num', num_cols),
    ('scaler', StandardScaler()),
    ('inputer', SimpleImputer(strategy='most_frequent'))
])

preprocessor = ColumnTransformer(transformers=[
    ('cat_preprocess', cat_preprocessor, cat_cols),
    ('num_preprocess', num_preprocessor, num_cols)
    ], 
    remainder='passthrough'
)

In [31]:
test_df.to_csv("cleaned_test_data.csv", index=False)
print("test dataset is ready to download")

test dataset is ready to download
